In [71]:
#Importamos las librerias

# Importamos la librería NumPy, utilizamos alias como np.
# Se utiliza para trabajar con arreglos, matrices y operaciones matemáticas.
import numpy as np

# Importamos la librería Pandas,utilizamos alias como pd.
# Se utiliza para manipular y analizar datos en tablas (DataFrames).
import pandas as pd

# Importamos la función create_engine desde SQLAlchemy.
# Esta función permite crear una conexión entre Python y una base de datos.
from sqlalchemy import create_engine

In [72]:
# 3.1 
# Importamos la librería csv para manejar reglas de lectura de archivos CSV
import csv

# Cargamos ventas_techventas.csv
df = pd.read_csv('ventas_techventas.csv',

    sep=',',                # Define que el separador de columnas es la coma                    
    encoding='latin1',      # Codificación de caracteres
    quoting=csv.QUOTE_NONE, # Le dice a pandas que NO intente interpretar comillas como delimitadores
    # parse_dates convierte 'fecha' de texto a tipo datetime
    parse_dates=['fecha']   # Columna a interpretar como fecha
)

df.head()


,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2,1200.0,0.10,Ana Lï¿½ï¿
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15,25.0,0.00,Carlos Ruiz
2,"""1003",2024-01-10,C003,DataSolutions,Centro,"Monitor 27""""",Electronica,5,350.0,0.05,"Ana Lï¿½ï¿"""
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3,1200.0,0.15,Carlos Ruiz


## Exploración inicial de los datos

Inicialmente se realizó una revisión general del conjunto de datos para identificar posibles problemas de calidad.

Durante esta exploración se encontraron los siguientes inconvenientes:

- Algunos registros fueron cargados incorrectamente, provocando que toda la información de la fila quedara almacenada en la columna order_id, mientras que las demás columnas aparecieran con valores nulos.
- La columna vendedor presenta errores de codificación en algunos registros, mostrando caracteres incorrectos como Ana Lï¿½ï¿.
- Se identificaron valores nulos en diferentes columnas debido a los errores de formato presentes en el archivo CSV.



In [73]:
# .info() es el método más completo para una auditoría rápida
# Muestra tipo de dato, cuántos no-nulos hay y memoria usada
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         60 non-null     str           
 1   fecha            60 non-null     datetime64[us]
 2   cliente_id       60 non-null     str           
 3   cliente_nombre   60 non-null     str           
 4   region           60 non-null     str           
 5   producto         59 non-null     str           
 6   categoria        60 non-null     str           
 7   cantidad         60 non-null     int64         
 8   precio_unitario  60 non-null     float64       
 9   descuento        60 non-null     float64       
 10  vendedor         60 non-null     str           
dtypes: datetime64[us](1), float64(2), int64(1), str(7)
memory usage: 5.3 KB


In [74]:
# Estadísticas descriptivas para columnas numéricas
df.describe()

,fecha,cantidad,precio_unitario,descuento
count,60,60.000000,60.000000,60.000000
mean,2024-03-27 00:24:00,9.766667,378.250000,0.055000
min,2024-01-05 00:00:00,-1.000000,25.000000,0.000000
25%,2024-02-14 06:00:00,3.000000,92.500000,0.000000
50%,2024-03-26 12:00:00,5.500000,120.000000,0.050000
75%,2024-05-07 18:00:00,15.750000,350.000000,0.100000
max,2024-06-18 00:00:00,35.000000,1200.000000,0.200000
std,NaN,8.857376,448.815602,0.058004


## Observaciones de calidad de los datos

Al realizar la exploración inicial del dataset se identificaron varios problemas de calidad de datos:

* La columna vendedo presenta errores de codificación en algunos registros, mostrando caracteres incorrectos como `Ana Lï¿½ï¿`.
* Se identificaron valores nulos en algunas columnas, lo que puede afectar los análisis posteriores.
* Existen registros con cantidades negativas, las cuales no son coherentes para una transacción de venta.

Antes de realizar los análisis con NumPy, es necesario corregir o eliminar estos registros para garantizar la calidad y consistencia de los resultados.

In [75]:
#Creamos una copia para nuestro archivo .csv, el original queda intacto
df_1 = df.copy()

print("Copia creada")

Copia creada


In [76]:
# 3.2 · Limpieza — detectar y tratar errores
# #Eliminamos los " que quedaron es nuestros df, para limpiar los datos 
df_1 = df_1.replace('"', '', regex=True)

df_1.head()

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2,1200.0,0.10,Ana Lï¿½ï¿
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15,25.0,0.00,Carlos Ruiz
2,1003,2024-01-10,C003,DataSolutions,Centro,Monitor 27,Electronica,5,350.0,0.05,Ana Lï¿½ï¿
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3,1200.0,0.15,Carlos Ruiz


In [77]:
#Verificamos los duplicados en la columna  order_id ya que debe tener un id único
df_1.duplicated(subset=['order_id']).any()



np.False_

In [78]:
# buscamos cual es el producto nulo en el df en la columna producto
df_1[df_1['producto'].isna()]

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
42,1043,2024-05-01,C010,DeltaOps,Norte,NaN,Electronica,2,350.0,0.05,Maria Torres


In [79]:
#Rellenamos los nulos
df_1['producto'] = df_1['producto'].fillna('Desconocido')

#Verificamos 
df_1[df_1['order_id'] == '1043' ]

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
42,1043,2024-05-01,C010,DeltaOps,Norte,Desconocido,Electronica,2,350.0,0.05,Maria Torres


In [80]:
# Convertimos order_id a entero ya que esta en String
df_1['order_id'] = df_1['order_id'].astype(int)

#verificamos
df_1.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         60 non-null     int64         
 1   fecha            60 non-null     datetime64[us]
 2   cliente_id       60 non-null     str           
 3   cliente_nombre   60 non-null     str           
 4   region           60 non-null     str           
 5   producto         60 non-null     str           
 6   categoria        60 non-null     str           
 7   cantidad         60 non-null     int64         
 8   precio_unitario  60 non-null     float64       
 9   descuento        60 non-null     float64       
 10  vendedor         60 non-null     str           
dtypes: datetime64[us](1), float64(2), int64(2), str(6)
memory usage: 5.3 KB


In [81]:
#Analizamos los negativos de cantidad
df_1[df_1['cantidad'] < 0]

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
47,1048,2024-05-15,C007,AlphaTech,Sur,Laptop Pro 15,Electronica,-1,1200.0,0.0,Ana Lï¿½ï¿


In [82]:
#corregimos cantidad negativa
df_1['cantidad'] = df_1['cantidad'].replace(-1,1)

#Verificamos 
df_1[df_1['order_id'] == 1048 ]

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
47,1048,2024-05-15,C007,AlphaTech,Sur,Laptop Pro 15,Electronica,1,1200.0,0.0,Ana Lï¿½ï¿


In [83]:
# Filtra las filas que contienen caracteres raros (cualquier cosa que no sea A-Z, a-z, 0-9 o espacios)
df_1[df_1['vendedor'].str.contains(r'[^a-zA-Z0-9\s]', regex=True, na=False)]


,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2,1200.0,0.10,Ana Lï¿½ï¿
2,1003,2024-01-10,C003,DataSolutions,Centro,Monitor 27,Electronica,5,350.0,0.05,Ana Lï¿½ï¿
6,1007,2024-01-20,C002,Innovatek,Sur,Monitor 27,Electronica,2,350.0,0.05,Ana Lï¿½ï¿
10,1011,2024-02-02,C001,TechCorp SA,Norte,Monitor 27,Electronica,4,350.0,0.05,Ana Lï¿½ï¿
15,1016,2024-02-15,C010,DeltaOps,Norte,Laptop Pro 15,Electronica,2,1200.0,0.20,Ana Lï¿½ï¿
19,1020,2024-02-25,C003,DataSolutions,Centro,Laptop Pro 15,Electronica,2,1200.0,0.10,Ana Lï¿½ï¿
23,1024,2024-03-10,C010,DeltaOps,Norte,Monitor 27,Electronica,3,350.0,0.05,Ana Lï¿½ï¿
27,1028,2024-03-20,C007,AlphaTech,Sur,Router WiFi 6,Redes,6,120.0,0.10,Ana Lï¿½ï¿
31,1032,2024-04-02,C009,GammaDevs,Oeste,SSD 1TB,Almacenamiento,22,95.0,0.10,Ana Lï¿½ï¿
35,1036,2024-04-12,C005,Sistemas Globales,Oeste,Monitor 27,Electronica,4,350.0,0.05,Ana Lï¿½ï¿


In [84]:
#corregimos nombres 
df_1['vendedor'] = df_1['vendedor'].replace('Ana Lï¿½ï¿','Ana Lopez')

#Verificamos 
df_1.head()

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2,1200.0,0.10,Ana Lopez
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15,25.0,0.00,Carlos Ruiz
2,1003,2024-01-10,C003,DataSolutions,Centro,Monitor 27,Electronica,5,350.0,0.05,Ana Lopez
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3,1200.0,0.15,Carlos Ruiz


## Ahora tenemos nuestros datos limpios

In [85]:
#Feature Engineering
#Valor total de la venta antes del descuento
df['ingreso_bruto'] = df['cantidad'] * df['precio_unitario']

df['ingreso_bruto'].head()


0    2400.0
1     375.0
2    1750.0
3     850.0
4    3600.0
Name: ingreso_bruto, dtype: float64

In [86]:
#Valor final de la venta después del descuento
df['ingreso_neto'] =df['ingreso_bruto'] * (1 - df['descuento'])

df['ingreso_neto'].head()

0    2160.0
1     375.0
2    1662.5
3     765.0
4    3060.0
Name: ingreso_neto, dtype: float64

In [87]:
#Extrae el año y mes de la fecha
df['mes'] = df['fecha'].dt.to_period('M').astype(str)

df['mes'].head()

0    2024-01
1    2024-01
2    2024-01
3    2024-01
4    2024-01
Name: mes, dtype: str

In [88]:
# Agruoamos por total de ventas por mes (ingreso neto) y numero de ventas (ventas) por mes 
#¿Cuánto se vendió por mes?
# ¿Cuántas ventas hubo por mes?
resumen = df.groupby('mes').agg(
    total=('ingreso_neto','sum'), 
    ventas=('order_id','count')
    ).reset_index() #convertir la columna índice de un DataFrame (creada tras agrupar los datos) en una columna normal del DataFrame.

print(resumen)


       mes     total  ventas
0  2024-01  13773.50      10
1  2024-02  13050.00      10
2  2024-03  14454.00      11
3  2024-04  16689.00      11
4  2024-05  11387.25      11
5  2024-06  11209.25       7


In [89]:
# Crear tabla de metas mensuales

metas = pd.DataFrame({
    'mes': ['2024-01', '2024-02', '2024-03'],
    'meta': [50000, 60000, 70000]
})

# Asegurar mismo tipo de dato (string)
metas['mes'] = metas['mes'].astype(str)


# Unir tablas
resumen_metas = pd.merge(
    resumen,
    metas,
    on='mes',  # Une ambas tablas usando la columna
    how='left' # Mantén todos los meses de la tabla resumen aunque no tengan meta
)

# Calcular porcentaje de cumplimiento
resumen_metas['cumplimiento_%'] = (
    resumen_metas['total']
    / resumen_metas['meta']
) * 100 # Menos de 100% meta no cumplida
# 100% meta cumplida
# Más de 100% meta superada

#Redondeamos a un decimal y agregamos "%"
resumen_metas['cumplimiento_%'] = resumen_metas['cumplimiento_%'].round(1).astype(str) + '%'

print(resumen_metas)

       mes     total  ventas     meta cumplimiento_%
0  2024-01  13773.50      10  50000.0          27.5%
1  2024-02  13050.00      10  60000.0          21.8%
2  2024-03  14454.00      11  70000.0          20.6%
3  2024-04  16689.00      11      NaN            NaN
4  2024-05  11387.25      11      NaN            NaN
5  2024-06  11209.25       7      NaN            NaN


In [90]:
# Crear conexión SQLite
engine = create_engine('sqlite:///techventas.db')

# Exportar DataFrame
df_1.to_sql(
    'ventas',
    engine,
    if_exists='replace',
    index=False
)

# Verificar leyendo nuevamente desde SQLite
verificacion = pd.read_sql(
    'SELECT COUNT(*) AS total_registros FROM ventas',
    engine
)

# SI nos quedo bien nos debe dar el umero de dtos de nuestra base de 
# datos (60)
print(verificacion)

   total_registros
0               60
